# 01. GitHub Archive 일별 집계 추출

GitHub Archive(BigQuery)에서 일별 이벤트를 **actor × repo × type** 기준으로 GROUP BY 집계한 뒤 parquet으로 저장합니다.

원본 테이블은 하루에 수천만~1억 row인데, 집계하면 150~250만 row로 줄어듭니다.

## 1. 설정

In [1]:
import os
from pathlib import Path
from datetime import date

from gharchive.client import create_client, get_logger
from gharchive.extract import dry_run, extract_single_day, extract_date_range, QUERY_TEMPLATE
from gharchive.transform import optimize_types

KEY_PATH = os.environ.get("GCP_KEY_PATH", "gcp-key.json")
OUTPUT_DIR = Path("../../data/daily_agg")

client = create_client(KEY_PATH)
logger = get_logger("extract")

print("Client ready:", client.project)

Client ready: bda-coai


## 2. 쿼리 설명 + Dry Run

집계 쿼리는 아래와 같습니다:

```sql
SELECT
    actor.id AS actor_id,
    repo.id AS repo_id,
    type,
    COUNT(*) AS cnt
FROM `githubarchive.day.{date_str}`
GROUP BY 1, 2, 3
```

- **actor.id**: 이벤트를 발생시킨 GitHub 유저 ID
- **repo.id**: 대상 repo ID
- **type**: 이벤트 종류 (PushEvent, WatchEvent, ForkEvent 등)
- **cnt**: 해당 조합의 이벤트 횟수

먼저 dry run으로 비용을 확인합니다.

In [2]:
test_date = "20260314"
info = dry_run(client, test_date)

print(f"Bytes processed: {info['bytes_processed'] / 1024**3:.2f} GB")
print(f"Estimated cost:  ${info['estimated_cost_usd']:.4f}")

Bytes processed: 0.10 GB
Estimated cost:  $0.0006


## 3. 1일 테스트 추출

실제로 1일분을 쿼리해서 row 수와 parquet 크기를 확인합니다.

In [3]:
df_test = extract_single_day(client, test_date)
df_test = optimize_types(df_test)

print(f"Rows: {len(df_test):,}")
print(f"Columns: {list(df_test.columns)}")
print(f"Dtypes:\n{df_test.dtypes}")
df_test.head()

Rows: 1,286,115
Columns: ['actor_id', 'repo_id', 'type', 'cnt']
Dtypes:
actor_id       Int32
repo_id        Int32
type        category
cnt            Int32
dtype: object


,actor_id,repo_id,type,cnt
0,30857654,1181908084,PublicEvent,1
1,35613825,1151667771,CommitCommentEvent,23
2,35613825,1151287497,CommitCommentEvent,7
3,36306781,1181648930,GollumEvent,5
4,41898282,912443158,CommitCommentEvent,6


In [4]:
# parquet 크기 확인
test_path = OUTPUT_DIR / "_test.parquet"
test_path.parent.mkdir(parents=True, exist_ok=True)
df_test.to_parquet(test_path, index=False)

size_mb = test_path.stat().st_size / 1024**2
print(f"Parquet size: {size_mb:.1f} MB")

# 테스트 파일 정리
test_path.unlink()

Parquet size: 11.1 MB


## 4. 28일 배치 다운로드

2026-02-15 ~ 2026-03-14 (28일) 데이터를 다운로드합니다.

이미 존재하는 파일은 자동으로 skip합니다.

In [5]:
START_DATE = date(2026, 2, 15)
END_DATE = date(2026, 3, 14)

saved_files = extract_date_range(client, START_DATE, END_DATE, OUTPUT_DIR, logger)
print(f"\nTotal files: {len(saved_files)}")

18:15:03 INFO Extracting 20260215 ...


18:15:08 INFO   → 1,186,753 rows, 13232 KB


18:15:08 INFO Extracting 20260216 ...


18:15:11 INFO   → 1,380,731 rows, 15360 KB


18:15:11 INFO Extracting 20260217 ...


...

18:16:52 INFO Extracting 20260313 ...


18:16:56 INFO   → 1,390,849 rows, 15691 KB


18:16:56 INFO Extracting 20260314 ...


18:16:59 INFO   → 1,286,115 rows, 14424 KB



Total files: 28


## 5. 검증

In [6]:
import pandas as pd

parquet_files = sorted(OUTPUT_DIR.glob("*.parquet"))
print(f"Parquet files: {len(parquet_files)}")

# 전체 로드
df_all = pd.read_parquet(OUTPUT_DIR)
df_all = optimize_types(df_all)

print(f"Total rows:     {len(df_all):,}")
print(f"Unique actors:  {df_all['actor_id'].nunique():,}")
print(f"Unique repos:   {df_all['repo_id'].nunique():,}")
print(f"Event types:    {sorted(df_all['type'].unique())}")
print(f"\nMemory usage:   {df_all.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Parquet files: 28


Total rows:     38,820,324


Unique actors:  6,097,393


Unique repos:   10,369,896
Event types:    ['CommitCommentEvent', 'CreateEvent', 'DeleteEvent', 'DiscussionEvent', 'ForkEvent', 'GollumEvent', 'IssueCommentEvent', 'IssuesEvent', 'MemberEvent', 'PublicEvent', 'PullRequestEvent', 'PullRequestReviewCommentEvent', 'PullRequestReviewEvent', 'PushEvent', 'ReleaseEvent', 'WatchEvent']

Memory usage:   592.4 MB
